# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** EU General Data Protection Regulation (GDPR, Regulation (EU) 2016/679) — an 88-page legal text full of
general rules, exceptions ("shall not apply where…"), prohibitions and role limits.

**What this notebook does**

| Step | What happens |
|---|---|
| 0 | Install libraries and set config |
| 1 | Download the PDF and extract text page by page |
| 2 | Split the text into overlapping chunks (same chunks for both pipelines) |
| 3 | **Pipeline 1 (Baseline):** bi-encoder embeddings → FAISS inner-product index → Top-10 → Top-3 to LLM |
| 4 | **Pipeline 2 (Re-rank):** same Top-10 → cross-encoder scores every (query, chunk) pair → re-sort → Top-3 to LLM |
| 5 | LLM answer generation with an identical prompt for both pipelines |
| 6 | Failure-mode stress tests with automatic rank-shift tracking |
| 7 | Side-by-side LLM answers (Answer 1 vs Answer 2) |
| 8 | Latency profiling + a scaling experiment (cross-encoder over the whole corpus) |
| 9 | Save all results to files for the report |

**How to run:** open in Google Colab → *Runtime → Run all*. For the LLM, add a Colab secret named `GEMINI_API_KEY`
(🔑 icon on the left). Without a key, the notebook falls back to a small local open-weights model (Qwen2.5-1.5B-Instruct).

## Step 0 — Install libraries and set configuration

In [1]:
# sentence-transformers: bi-encoder + cross-encoder models
# faiss-cpu: vector store for the dense index
# pypdf: PDF text extraction
# google-genai: Gemini API client (LLM)
# accelerate: needed only for the local fallback LLM (device_map="auto")
%pip install -q sentence-transformers faiss-cpu pypdf google-genai pandas accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\guruk\Documents\Desktop_files\LMA_RAG\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os, re, io, time, json, textwrap, statistics
import numpy as np
import pandas as pd
import requests
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

pd.set_option("display.max_colwidth", 120)

# ---------------- Configuration ----------------
PDF_URLS = [  # official PDF of the GDPR (tried in order)
    # EU Publications Office (same Official Journal PDF; EUR-Lex often blocks scripted downloads)
    "https://op.europa.eu/o/opportal-service/download-handler?identifier=3e485e15-11bd-11e6-ba9a-01aa75ed71a1&format=pdf&language=en&productionSystem=cellar",
    "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
    "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679&from=EN",
]
PDF_PATH = "gdpr.pdf"

CHUNK_WORDS   = 200   # words per chunk
CHUNK_OVERLAP = 40    # words shared between neighbouring chunks
TOP_K   = 10          # candidates retrieved by the bi-encoder
FINAL_K = 3           # chunks given to the LLM

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# LLM: Gemini if a key is available, otherwise a local open-weights model
GEMINI_MODEL = "gemini-3.5-flash-lite"      # free tier: ~20 requests/day per model; if quota runs out, try "gemini-3.1-flash-lite"
LOCAL_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## Step 1 — Load the document

Download the GDPR PDF and extract the text of each page. If the download is blocked, the cell asks you to upload the PDF manually.

In [3]:
def download_pdf(urls, path):
    headers = {"User-Agent": "Mozilla/5.0 (RAG assignment notebook)"}
    for url in urls:
        try:
            r = requests.get(url, headers=headers, timeout=60)
            if r.ok and r.content[:4] == b"%PDF":     # make sure we got a real PDF, not an HTML page
                open(path, "wb").write(r.content)
                print(f"Downloaded {len(r.content)/1e6:.1f} MB from {url}")
                return True
            print(f"Not a PDF from {url} (status {r.status_code})")
        except Exception as e:
            print(f"Failed {url}: {e}")
    return False

if not os.path.exists(PDF_PATH) and not download_pdf(PDF_URLS, PDF_PATH):
    # Fallback: upload the GDPR PDF by hand (Colab only)
    try:
        from google.colab import files
        uploaded = files.upload()
        PDF_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Download failed. Save the GDPR PDF next to this notebook as '{PDF_PATH}' and re-run.")

reader = PdfReader(PDF_PATH)
pages = []
for i, page in enumerate(reader.pages):
    t = page.extract_text() or ""
    t = re.sub(r"-\n(\w)", r"\1", t)           # join words hyphenated across line breaks
    t = t.replace("\xa0", " ")                  # non-breaking spaces -> normal spaces
    t = re.sub(r"\s+", " ", t).strip()          # collapse all whitespace
    pages.append({"page": i + 1, "text": t})

print("Pages:", len(pages))
print("Total words:", sum(len(p["text"].split()) for p in pages))
print(pages[40]["text"][:500])

Pages: 88
Total words: 55812
(d) where the processing is based on point (f) of Article 6(1), the legitimate interests pursued by the controller or by a third party; (e) the recipients or categories of recipients of the personal data, if any; (f) where applicable, the fact that the controller intends to transfer personal data to a third country or international organisation and the existence or absence of an adequacy decision by the Commission, or in the case of transfers referred to in Article 46 or 47, or the second subpar


## Step 2 — Chunk the document

Fixed-size word windows with overlap. Every chunk keeps the page it starts on.
Both pipelines use **exactly this same list of chunks**.

In [4]:
def chunk_pages(pages, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    # Build one long word list, remembering which page each word came from
    words, word_page = [], []
    for p in pages:
        w = p["text"].split()
        words += w
        word_page += [p["page"]] * len(w)

    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        if len(piece) < 30:            # skip tiny leftover at the end
            break
        chunks.append({
            "chunk_id": len(chunks),
            "page": word_page[start],
            "text": " ".join(piece),
        })
    return chunks

chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
print("Number of chunks:", len(chunks))
print("Example chunk:", chunks[100])

Number of chunks: 349
Example chunk: {'chunk_id': 100, 'page': 21, 'text': 'should be able to make use of approved binding corporate rules for its international transfers from the Union to organisations within the same group of undertakings, or group of enterprises engaged in a joint economic activity, provided that such corporate rules include all essential principles and enforceable rights to ensure appropriate safeguards for transfers or categories of transfers of personal data. (111) Provisions should be made for the possibility for transfers in certain circumstances where the data subject has given his or her explicit consent, where the transfer is occasional and necessary in relation to a contract or a legal claim, regardless of whether in a judicial procedure or whether in an administrative or any out-of-court procedure, including procedures before regulatory bodies. Provision should also be made for the possibility for transfers where important grounds of public interest laid d

## Step 3 — Pipeline 1: Bi-Encoder + vector store (Baseline)

The bi-encoder turns **each chunk on its own** into a vector, and the query into another vector.
Similarity = cosine similarity (inner product of L2-normalised vectors) in a FAISS `IndexFlatIP`.

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=True).astype("float32")
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])   # inner product on normalised vectors == cosine
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, indexing took {index_time:.2f}s (one-time cost)")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Indexed 349 chunks, dim=384, indexing took 5.95s (one-time cost)


In [6]:
def baseline_retrieve(query, k=TOP_K):
    # Embed the query and return the top-k chunks by cosine similarity.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]))]

baseline_retrieve("What are the rights of the data subject?")[:3]

[{'chunk_id': 192, 'bi_rank': 1, 'bi_score': 0.7293280959129333},
 {'chunk_id': 55, 'bi_rank': 2, 'bi_score': 0.7212479710578918},
 {'chunk_id': 181, 'bi_rank': 3, 'bi_score': 0.7082312703132629}]

## Step 4 — Pipeline 2: Cross-Encoder re-ranking

The cross-encoder reads the query **and** the chunk together in one input (`[CLS] query [SEP] chunk`),
so every query word can attend to every chunk word. It outputs one relevance score per pair.
We only run it on the 10 candidates from Step 3.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE, max_length=512)

def rerank(query, candidates, final_k=FINAL_K):
    # Score all (query, chunk) pairs with the cross-encoder and re-sort descending.
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    out.sort(key=lambda x: x["ce_score"], reverse=True)
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

cands = baseline_retrieve("What are the rights of the data subject?")
pd.DataFrame(rerank("What are the rights of the data subject?", cands))

,chunk_id,bi_rank,bi_score,ce_score,ce_rank
0,181,3,0.708231,7.128688,1
1,56,10,0.679821,6.923499,2
2,62,8,0.691987,6.478381,3
3,240,5,0.700775,6.228701,4
4,180,7,0.696005,6.035712,5
5,326,9,0.690088,5.795056,6
6,57,4,0.707592,5.120713,7
7,192,1,0.729328,4.974197,8
8,55,2,0.721248,4.851007,9
9,141,6,0.697007,1.160383,10


## Step 5 — LLM answer generation (identical prompt for both pipelines)

Only the 3 context chunks differ between Answer 1 and Answer 2. Temperature is 0 so differences come from the context, not randomness.

In [8]:
PROMPT_TEMPLATE = '''You are a legal assistant answering questions about the EU GDPR.
Answer ONLY using the context below. Quote the article or paragraph if it is visible.
If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}
Answer (max 5 sentences):'''

def build_prompt(question, chunk_ids):
    context = "\n\n".join(f"[Chunk {cid}, page {chunks[cid]['page']}]\n{chunks[cid]['text']}" for cid in chunk_ids)
    return PROMPT_TEMPLATE.format(context=context, question=question)

# ---- choose the LLM backend ----
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata
    GEMINI_API_KEY = GEMINI_API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass

LLM_CACHE = {}        # identical prompt -> same answer at temperature 0, so do not pay for it twice
LLM_CALL_TIMES = []   # seconds per real (non-cached) LLM call, used in the latency table

if GEMINI_API_KEY:
    from google import genai
    from google.genai import types
    gclient = genai.Client(api_key=GEMINI_API_KEY)
    LLM_BACKEND = f"gemini:{GEMINI_MODEL}"

    def call_llm(prompt):
        for attempt in range(4):                    # simple retry for free-tier rate limits
            try:
                r = gclient.models.generate_content(
                    model=GEMINI_MODEL, contents=prompt,
                    # generous token cap: on "thinking" models, reasoning tokens count toward it
                    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=2048))
                if r.text:
                    return r.text.strip()
                print("Gemini returned no text, retrying")
            except Exception as e:
                msg = str(e)
                if "PerDay" in msg:                 # daily free-tier quota used up: retrying will not help
                    print(f"Gemini daily quota exhausted for {GEMINI_MODEL}. Change GEMINI_MODEL and re-run.")
                    return "[LLM call failed: daily quota]"
                print("Gemini error, retrying:", msg[:150])
                time.sleep(15 * (attempt + 1))
        return "[LLM call failed]"
else:
    from transformers import pipeline
    gen = pipeline("text-generation", model=LOCAL_MODEL, device_map="auto", torch_dtype="auto")
    LLM_BACKEND = f"local:{LOCAL_MODEL}"

    def call_llm(prompt):
        msgs = [{"role": "user", "content": prompt}]
        out = gen(msgs, max_new_tokens=300, do_sample=False)
        return out[0]["generated_text"][-1]["content"].strip()

def llm(prompt):
    if prompt not in LLM_CACHE:
        t0 = time.perf_counter()
        LLM_CACHE[prompt] = call_llm(prompt)
        LLM_CALL_TIMES.append(time.perf_counter() - t0)
    return LLM_CACHE[prompt]

print("LLM backend:", LLM_BACKEND)
print(llm("Reply with the single word: ready"))

LLM backend: gemini:gemini-3.5-flash-lite


ready


In [9]:
def pipeline1(query):
    # Baseline: bi-encoder top-10 -> take top-3 as is -> LLM
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)
    t_ret = time.perf_counter() - t0
    top3 = [c["chunk_id"] for c in cands[:FINAL_K]]
    t1 = time.perf_counter()
    answer = llm(build_prompt(query, top3))
    t_llm = time.perf_counter() - t1
    return {"top3": top3, "answer": answer, "retrieval_s": t_ret, "llm_s": t_llm, "cands": cands}

def pipeline2(query):
    # Re-rank: bi-encoder top-10 -> cross-encoder re-sort -> top-3 -> LLM
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)
    reranked = rerank(query, cands)
    t_ret = time.perf_counter() - t0
    top3 = [c["chunk_id"] for c in reranked[:FINAL_K]]
    t1 = time.perf_counter()
    answer = llm(build_prompt(query, top3))
    t_llm = time.perf_counter() - t1
    return {"top3": top3, "answer": answer, "retrieval_s": t_ret, "llm_s": t_llm, "cands": reranked}

## Step 6 — Failure-mode stress tests

Each query is written the way a normal person would ask it (everyday words), while the GDPR answer uses legal wording,
or sits in an exception paragraph next to a more "obvious" general rule.

For every query, `evidence` holds short phrases that appear **only in the clause that truly answers it**.
Any chunk containing one of them is a *gold chunk*. We then compare the gold chunk's rank in the baseline vs. after re-ranking.

| Category | What the trap is |
|---|---|
| Keyword trap | Query shares words with an irrelevant passage (e.g. "penalty" → Art. 84 *Penalties*), while the answer uses other words ("administrative fines") |
| Exception / corner case | Bi-encoder finds the general rule but misses the "shall not apply…" paragraph |
| Negation / constraint | "NOT required", "prohibited", "shall not be dismissed" — embeddings blur positive and negative |
| Role limit | What a specific role (e.g. the DPO) may / may not do |

In [10]:
QUERIES = [
    {"id": "Q1", "category": "Keyword trap",
     "query": "What is the maximum penalty amount a company can be charged for violating this regulation?",
     "evidence": ["20 000 000 EUR", "4 % of the total worldwide annual turnover"],
     "note": "Art. 83(5) uses 'administrative fines'; Art. 84 is literally titled 'Penalties' but only says Member States set rules."},
    {"id": "Q2", "category": "Keyword trap",
     "query": "How fast must a company report a hack to the regulator?",
     "evidence": ["not later than 72 hours"],
     "note": "GDPR says 'personal data breach' and 'supervisory authority', never 'hack' or 'regulator'."},
    {"id": "Q3", "category": "Keyword trap",
     "query": "At what age can a kid sign up for a social media app without asking their parents?",
     "evidence": ["at least 16 years old", "not below 13 years"],
     "note": "Art. 8 says 'information society services', 'child', 'holder of parental responsibility'."},
    {"id": "Q4", "category": "Exception / corner case",
     "query": "Does a small business with only 50 employees still have to maintain records of its processing activities?",
     "evidence": ["fewer than 250 persons"],
     "note": "Art. 30(1) is the general rule; the exemption is in Art. 30(5) (with its own 'unless' conditions)."},
    {"id": "Q5", "category": "Exception / corner case",
     "query": "In what situations can a company refuse my request to delete my personal data?",
     "evidence": ["shall not apply to the extent that processing is necessary"],
     "note": "Right to erasure Art. 17(1) vs. the exceptions in Art. 17(3)."},
    {"id": "Q6", "category": "Exception / corner case",
     "query": "Does the GDPR apply to a person keeping a list of friends' phone numbers and birthdays?",
     "evidence": ["purely personal or household activity"],
     "note": "Household exemption, Art. 2(2)(c)."},
    {"id": "Q7", "category": "Negation / constraint",
     "query": "In which situations is a company NOT required to tell affected people about a data breach?",
     "evidence": ["shall not be required if any of the following conditions"],
     "note": "Art. 34(1) general duty vs. Art. 34(3) conditions where it is not required."},
    {"id": "Q8", "category": "Negation / constraint",
     "query": "Is it prohibited to process data about someone's religion or health?",
     "evidence": ["revealing racial or ethnic origin, political opinions, religious or philosophical beliefs"],
     "note": "Art. 9(1) prohibition of special categories."},
    {"id": "Q9", "category": "Negation / role limit",
     "query": "Can the company fire its data protection officer for doing their job?",
     "evidence": ["shall not be dismissed or penalised"],
     "note": "Art. 38(3) protects the DPO."},
    {"id": "Q10", "category": "Negation / constraint",
     "query": "Is a company allowed to make important decisions about me using only a computer algorithm?",
     "evidence": ["based solely on automated processing"],
     "note": "Art. 22 automated decision-making."},
]

def squash(s):
    # lowercase and remove ALL whitespace so PDF spacing quirks do not break matching
    return re.sub(r"\s+", "", s.lower())

squashed_chunks = [squash(t) for t in chunk_texts]

def gold_chunks(evidence):
    ev = [squash(e) for e in evidence]
    return {i for i, t in enumerate(squashed_chunks) if any(e in t for e in ev)}

def best_rank(ranked_ids, gold):
    # 1-based rank of the first gold chunk in a ranked list, or None if absent
    for r, cid in enumerate(ranked_ids):
        if cid in gold:
            return r + 1
    return None

# Gold chunk ids per query (sanity check: every query should have at least one)
for q in QUERIES:
    q["gold"] = gold_chunks(q["evidence"])
    print(q["id"], "gold chunks:", sorted(q["gold"]), "pages:", sorted({chunks[g]['page'] for g in q['gold']}))

Q1 gold chunks: [332, 333, 334] pages: [82, 83]
Q2 gold chunks: [79, 223] pages: [16, 52]
Q3 gold chunks: [174] pages: [37]
Q4 gold chunks: [220, 221] pages: [51]
Q5 gold chunks: [197] pages: [44]
Q6 gold chunks: [12, 155] pages: [3, 32]
Q7 gold chunks: [225, 226] pages: [52, 53]
Q8 gold chunks: [175] pages: [38]
Q9 gold chunks: [236, 237] pages: [55, 56]
Q10 gold chunks: [64, 203, 263, 264] pages: [13, 46, 63]


In [11]:
rows, details = [], {}
for q in QUERIES:
    cands = baseline_retrieve(q["query"])
    reranked = rerank(q["query"], cands)
    base_ids = [c["chunk_id"] for c in cands]
    rr_ids   = [c["chunk_id"] for c in reranked]
    b, r = best_rank(base_ids, q["gold"]), best_rank(rr_ids, q["gold"])
    if b is None:
        verdict = "Gold not in top-10 (re-ranker cannot recover it)"
    elif b <= FINAL_K and r <= FINAL_K:
        verdict = "Both pipelines OK"
    elif b > FINAL_K and r <= FINAL_K:
        verdict = "RE-RANK FIXED IT"
    elif b <= FINAL_K and r > FINAL_K:
        verdict = "Re-rank made it worse"
    else:
        verdict = "Both miss top-3"
    rows.append({"id": q["id"], "category": q["category"], "query": q["query"],
                 "baseline_rank": b, "reranked_rank": r, "verdict": verdict})
    details[q["id"]] = reranked   # keeps bi_rank, bi_score, ce_rank, ce_score

rank_df = pd.DataFrame(rows)
rank_df

,id,category,query,baseline_rank,reranked_rank,verdict
0,Q1,Keyword trap,What is the maximum penalty amount a company can be charged for violating this regulation?,2.0,1.0,Both pipelines OK
1,Q2,Keyword trap,How fast must a company report a hack to the regulator?,3.0,3.0,Both pipelines OK
2,Q3,Keyword trap,At what age can a kid sign up for a social media app without asking their parents?,1.0,1.0,Both pipelines OK
3,Q4,Exception / corner case,Does a small business with only 50 employees still have to maintain records of its processing activities?,NaN,NaN,Gold not in top-10 (re-ranker cannot recover it)
4,Q5,Exception / corner case,In what situations can a company refuse my request to delete my personal data?,NaN,NaN,Gold not in top-10 (re-ranker cannot recover it)
5,Q6,Exception / corner case,Does the GDPR apply to a person keeping a list of friends' phone numbers and birthdays?,10.0,1.0,RE-RANK FIXED IT
6,Q7,Negation / constraint,In which situations is a company NOT required to tell affected people about a data breach?,1.0,1.0,Both pipelines OK
7,Q8,Negation / constraint,Is it prohibited to process data about someone's religion or health?,4.0,1.0,RE-RANK FIXED IT
8,Q9,Negation / role limit,Can the company fire its data protection officer for doing their job?,1.0,1.0,Both pipelines OK
9,Q10,Negation / constraint,Is a company allowed to make important decisions about me using only a computer algorithm?,1.0,1.0,Both pipelines OK


### Rank-shift detail for one query

Shows all 10 candidates: original bi-encoder rank → cross-encoder rank. ⭐ marks a gold chunk.
Change `QID` to look at any query.

In [12]:
def show_shift(qid):
    q = next(x for x in QUERIES if x["id"] == qid)
    print(f"{qid}: {q['query']}\nWhy it is hard: {q['note']}\n")
    df = pd.DataFrame([{
        "chunk": c["chunk_id"], "page": chunks[c["chunk_id"]]["page"],
        "gold": "⭐" if c["chunk_id"] in q["gold"] else "",
        "bi_rank": c["bi_rank"], "ce_rank": c["ce_rank"],
        "shift": f"#{c['bi_rank']} → #{c['ce_rank']}",
        "bi_score": round(c["bi_score"], 3), "ce_score": round(c["ce_score"], 2),
        "preview": chunks[c["chunk_id"]]["text"][:110] + "…",
    } for c in details[qid]])
    return df.sort_values("ce_rank")

QID = "Q1"
show_shift(QID)

Q1: What is the maximum penalty amount a company can be charged for violating this regulation?
Why it is hard: Art. 83(5) uses 'administrative fines'; Art. 84 is literally titled 'Penalties' but only says Member States set rules.



,chunk,page,gold,bi_rank,ce_rank,shift,bi_score,ce_score,preview
0,332,82,⭐,2,1,#2 → #1,0.548,1.01,"intentionally or negligently, for the same or linked processing operations, infringes several provisions of th…"
1,133,27,,1,2,#1 → #2,0.582,-0.51,"order to strengthen the enforcement of the rules of this Regulation, penalties including administrative fines …"
2,134,27,,3,3,#3 → #3,0.545,-0.95,"with the general principles of Union law and the Charter, including effective judicial protection and due proc…"
3,135,27,,5,4,#5 → #4,0.518,-2.22,12 December 2012 on jurisdiction and the recognition and enforcement of judgments in civil and commercial matt…
4,335,83,,4,5,#4 → #5,0.540,-3.11,"imposed by supervisory authorities. In any event, the fines imposed shall be effective, proportionate and diss…"
5,137,28,,7,6,#7 → #6,0.479,-3.65,administrative fines imposed by supervisory authorities. Therefore the competent national courts should take i…
6,330,82,,10,7,#10 → #7,0.459,-5.18,"5 and 6 shall in each individual case be effective, proportionate and dissuasive. 2. Administrative fines shal…"
7,329,81,,6,8,#6 → #8,0.488,-5.90,controller or processor shall be held liable for the entire damage in order to ensure effective compensation o…
8,334,83,⭐,8,9,#8 → #9,0.479,-7.25,"or in the case of an undertaking, up to 4 % of the total worldwide annual turnover of the preceding financial …"
9,331,82,,9,10,#9 → #10,0.475,-8.41,to Articles 25 and 32; (e) any relevant previous infringements by the controller or processor; (f) the degree …


In [13]:
# Print the rank movement of every gold chunk, ready to copy into the report
for q in QUERIES:
    for c in details[q["id"]]:
        if c["chunk_id"] in q["gold"]:
            print(f"{q['id']}: Chunk {c['chunk_id']} (page {chunks[c['chunk_id']]['page']}) "
                  f"moved from Rank #{c['bi_rank']} → Rank #{c['ce_rank']}")

Q1: Chunk 332 (page 82) moved from Rank #2 → Rank #1
Q1: Chunk 334 (page 83) moved from Rank #8 → Rank #9
Q2: Chunk 79 (page 16) moved from Rank #3 → Rank #3
Q2: Chunk 223 (page 52) moved from Rank #6 → Rank #4
Q3: Chunk 174 (page 37) moved from Rank #1 → Rank #1
Q6: Chunk 12 (page 3) moved from Rank #10 → Rank #1
Q7: Chunk 226 (page 53) moved from Rank #5 → Rank #1
Q7: Chunk 225 (page 52) moved from Rank #1 → Rank #8
Q8: Chunk 175 (page 38) moved from Rank #4 → Rank #1
Q9: Chunk 237 (page 56) moved from Rank #1 → Rank #1
Q9: Chunk 236 (page 55) moved from Rank #3 → Rank #2
Q10: Chunk 64 (page 13) moved from Rank #6 → Rank #1
Q10: Chunk 203 (page 46) moved from Rank #1 → Rank #2


## Step 7 — Compare LLM Answer 1 (baseline) vs. Answer 2 (re-ranked)

Runs both full pipelines for every query. Look for: hallucinated numbers, a general rule stated without its exception,
or "context does not answer this" in Answer 1 but a correct answer in Answer 2.

In [14]:
answers = []
for q in QUERIES:
    p1, p2 = pipeline1(q["query"]), pipeline2(q["query"])
    answers.append({
        "id": q["id"], "query": q["query"],
        "p1_top3": p1["top3"], "p2_top3": p2["top3"],
        "p1_has_gold": bool(set(p1["top3"]) & q["gold"]),
        "p2_has_gold": bool(set(p2["top3"]) & q["gold"]),
        "answer_1": p1["answer"], "answer_2": p2["answer"],
    })
    print("=" * 100)
    print(f"{q['id']} [{q['category']}] {q['query']}")
    print(f"\n--- Answer 1 (baseline, chunks {p1['top3']}, gold in context: {answers[-1]['p1_has_gold']}) ---")
    print(textwrap.fill(p1["answer"], 100))
    print(f"\n--- Answer 2 (re-ranked, chunks {p2['top3']}, gold in context: {answers[-1]['p2_has_gold']}) ---")
    print(textwrap.fill(p2["answer"], 100))
    if LLM_BACKEND.startswith("gemini"):
        time.sleep(4)   # stay under free-tier requests-per-minute

answers_df = pd.DataFrame(answers)

Q1 [Keyword trap] What is the maximum penalty amount a company can be charged for violating this regulation?

--- Answer 1 (baseline, chunks [133, 332, 134], gold in context: True) ---
Based on the provided context, infringements of certain provisions (such as the basic principles for
processing and conditions for consent) are subject to administrative fines up to 20 000 000 EUR, or
in the case of an undertaking, up to 4 % of the total worldwide annual turnover of the preceding
financial year, whichever is higher (Chunk 332, page 82). Additionally, infringements of other
specific provisions (such as the obligations of the controller and processor pursuant to Articles 8,
11, 25 to 39 and 42 and 43) are subject to administrative fines up to 10 000 000 EUR, or in the case
of an undertaking, up to 2 % of the total worldwide annual turnover of the preceding financial year,
whichever is higher (Chunk 332, page 82). If someone, intentionally or negligently, infringes
several provisions for th

Q2 [Keyword trap] How fast must a company report a hack to the regulator?

--- Answer 1 (baseline, chunks [232, 284, 79], gold in context: True) ---
Based on the provided context, as soon as the controller becomes aware that a personal data breach
(hack) has occurred, "the controller should notify the personal data breach to the supervisory
authority without undue delay and, where feasible, not later than 72 hours after having become aware
of it, unless the controller is able to demonstrate, in accordance with the accountability
principle, that the personal data breach is unlikely to result in a risk to the rights and freedoms
of natural persons" (Recital 85).

--- Answer 2 (re-ranked, chunks [224, 232, 79], gold in context: True) ---
Based on the provided context, the controller should notify the supervisory authority of a personal
data breach "without undue delay and, where feasible, not later than 72 hours after having become
aware of it" (Chunk 79, page 16). If the notification is 

Q3 [Keyword trap] At what age can a kid sign up for a social media app without asking their parents?

--- Answer 1 (baseline, chunks [174, 52, 30], gold in context: True) ---
Based on the provided context, the processing of a child's personal data in relation to the offer of
information society services directly to a child is lawful where the child is at least 16 years old.
As stated in Article 8(1), "Where the child is below the age of 16 years, such processing shall be
lawful only if and to the extent that consent is given or authorised by the holder of parental
responsibility over the child." However, "Member States may provide by law for a lower age for those
purposes provided that such lower age is not below 13 years."

--- Answer 2 (re-ranked, chunks [174, 52, 30], gold in context: True) ---
Based on the provided context, the processing of a child's personal data in relation to the offer of
information society services directly to a child is lawful where the child is at least 16 

Q4 [Exception / corner case] Does a small business with only 50 employees still have to maintain records of its processing activities?

--- Answer 1 (baseline, chunks [9, 218, 333], gold in context: False) ---
The provided context does not answer this.

--- Answer 2 (re-ranked, chunks [9, 218, 141], gold in context: False) ---
The provided context does not explicitly state whether a business with 50 employees must maintain
records. However, it mentions that "this Regulation includes a derogation for organisations with
fewer than 250 employees with regard to record-keeping" (Chunk 9, page 3). Furthermore, Article 30,
paragraph 1 states that "Each controller and, where applicable, the controller's representative,
shall maintain a record of processing activities under its responsibility" (Chunk 218, page 50).


Q5 [Exception / corner case] In what situations can a company refuse my request to delete my personal data?

--- Answer 1 (baseline, chunks [196, 195, 193], gold in context: False) ---
The provided context does not answer this.

--- Answer 2 (re-ranked, chunks [57, 58, 32], gold in context: False) ---
Based on the provided context, the retention of personal data remains lawful and is not required to
be erased where it is necessary "for exercising the right of" (the sentence is cut off in the text).
Additionally, a data subject has the right to be forgotten where retention infringes the Regulation
or Union or Member State law, and specifically where data are no longer necessary for the purposes
collected, where consent is withdrawn, where the data subject objects to processing, or where
processing does not otherwise comply with the Regulation. The provided context does not explicitly
list situations where a company can refuse a deletion request beyond stating when the right applies
and 

Q6 [Exception / corner case] Does the GDPR apply to a person keeping a list of friends' phone numbers and birthdays?

--- Answer 1 (baseline, chunks [22, 20, 337], gold in context: False) ---
The provided context does not answer this.

--- Answer 2 (re-ranked, chunks [12, 21, 20], gold in context: True) ---
Based on the provided context, the GDPR does not apply to the processing of personal data by a
natural person in the course of a purely personal or household activity with no connection to a
professional or commercial activity. As stated in Recital 18, "Personal or household activities
could include correspondence and the holding of addresses," which covers activities like keeping a
list of friends' phone numbers and birthdays. However, the text also notes that "this Regulation
applies to controllers or processors which provide the means for processing personal data for such
personal or household activities."


Q7 [Negation / constraint] In which situations is a company NOT required to tell affected people about a data breach?

--- Answer 1 (baseline, chunks [225, 224, 82], gold in context: True) ---
Based on the provided context, communication of a personal data breach to the data subject is not
required if the following condition is met:   * "(a) the controller has implemented appropriate
technical and organisational protection measures, and those" [Article 34(3)(a)]

--- Answer 2 (re-ranked, chunks [226, 79, 82], gold in context: True) ---
Based on the provided context, the communication of a personal data breach to the data subject is
not required if any of the following conditions are met:  1. As stated in Article 34(3)(a), "the
controller has implemented appropriate technical and organisational protection measures, and those
measures were applied to the personal data affected by the personal data breach, in particular those
that render the personal data unintelligible to any person who 

Q8 [Negation / constraint] Is it prohibited to process data about someone's religion or health?

--- Answer 1 (baseline, chunks [50, 56, 46], gold in context: False) ---
Based on the provided context, special categories of personal data (which include health and
potentially others) are generally subject to a prohibition on processing, but derogating from this
prohibition is allowed under certain conditions. Specifically, derogations are permitted when
provided for in Union or Member State law and subject to suitable safeguards, where it is in the
public interest to do so (such as in the field of employment law, social protection law, and health
purposes) or where the data subject gives explicit consent (Chunk 46). Additionally, the processing
of data concerning health for reasons of public interest should not result in personal data being
processed for other purposes by third parties such as employers or insurance and banking companies
(Chunk 50).

--- Answer 2 (re-ranked, chunks [175,

Q9 [Negation / role limit] Can the company fire its data protection officer for doing their job?

--- Answer 1 (baseline, chunks [237, 235, 236], gold in context: True) ---
No, the company cannot dismiss or penalise the data protection officer for performing their tasks.
As stated in Chunk 236 (Article 38, paragraph 3): "He or she shall not be dismissed or penalised by
the controller or the processor for performing his tasks."

--- Answer 2 (re-ranked, chunks [237, 236, 235], gold in context: True) ---
No, the company cannot dismiss or penalise the data protection officer for performing their tasks.
As stated in Article 38(3), "He or she shall not be dismissed or penalised by the controller or the
processor for performing his tasks."


Q10 [Negation / constraint] Is a company allowed to make important decisions about me using only a computer algorithm?

--- Answer 1 (baseline, chunks [203, 56, 201], gold in context: True) ---
According to Article 22(1), "The data subject shall have the right not to be subject to a decision
based solely on automated processing, including profiling, which produces legal effects concerning
him or her or similarly significantly affects him or her."   However, paragraph 1 does not apply if
the decision "(a) is necessary for entering into, or performance of, a contract between the data
subject and a data controller; (b) is authorised" (as stated in Article 22(2)).

--- Answer 2 (re-ranked, chunks [64, 203, 212], gold in context: True) ---
Based on the provided context, the data subject generally has the right not to be subject to a
decision based solely on automated processing, including profiling, which produces legal effects
concerning them or similarly significantly affects them. As sta

## Step 8 — Latency profiling

Each stage is timed separately, with a warm-up run first (model loading / first-call overhead excluded) and averaged over several runs.
LLM time is reported separately because it depends on the API/network and is the same work for both pipelines.

In [15]:
N_RUNS = 5
test_queries = [q["query"] for q in QUERIES]

def timeit(fn, runs=N_RUNS):
    fn()                                    # warm-up
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    return statistics.mean(ts), statistics.pstdev(ts)

lat = {"P1: embed query + FAISS top-10 (+ take top-3)": [],
       "Cross-encoder scoring of 10 pairs + sort": [],
       "P2: total retrieval (bi-encoder + re-rank)": []}
for qt in test_queries:
    cands = baseline_retrieve(qt)
    lat["P1: embed query + FAISS top-10 (+ take top-3)"].append(timeit(lambda: baseline_retrieve(qt))[0])
    lat["Cross-encoder scoring of 10 pairs + sort"].append(timeit(lambda: rerank(qt, cands))[0])
    lat["P2: total retrieval (bi-encoder + re-rank)"].append(timeit(lambda: rerank(qt, baseline_retrieve(qt)))[0])

latency_df = pd.DataFrame([{"stage": k, "mean_ms": round(statistics.mean(v), 2),
                            "min_ms": round(min(v), 2), "max_ms": round(max(v), 2)} for k, v in lat.items()])

# LLM latency: reuse the real (non-cached) calls timed in Step 7 instead of spending more API quota
llm_times = [s * 1000 for s in LLM_CALL_TIMES[1:]]   # [1:] skips the "ready" test call
latency_df.loc[len(latency_df)] = ["LLM generation (same for both)", round(statistics.mean(llm_times), 1),
                                   round(min(llm_times), 1), round(max(llm_times), 1)]
base = latency_df.loc[0, "mean_ms"]
latency_df["x_vs_baseline_retrieval"] = (latency_df["mean_ms"] / base).round(1)
print(f"Device: {DEVICE} | chunks: {len(chunks)} | runs per query: {N_RUNS} | LLM: {LLM_BACKEND}")
latency_df

Device: cpu | chunks: 349 | runs per query: 5 | LLM: gemini:gemini-3.5-flash-lite


,stage,mean_ms,min_ms,max_ms,x_vs_baseline_retrieval
0,P1: embed query + FAISS top-10 (+ take top-3),10.96,6.04,13.94,1.0
1,Cross-encoder scoring of 10 pairs + sort,303.25,199.87,379.33,27.7
2,P2: total retrieval (bi-encoder + re-rank),328.26,210.52,407.84,30.0
3,LLM generation (same for both),1637.40,722.60,4994.00,149.4


### Scaling experiment — why not run the cross-encoder on every chunk?

Time the cross-encoder over **all** chunks for one query and compare with scoring 10 candidates.
Then extrapolate to a large collection (each query would need one transformer pass per document).

In [16]:
qt = QUERIES[0]["query"]
all_pairs = [(qt, t) for t in chunk_texts]

t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32); t_all = time.perf_counter() - t0
t0 = time.perf_counter(); rerank(qt, baseline_retrieve(qt));              t_10  = time.perf_counter() - t0
t0 = time.perf_counter(); baseline_retrieve(qt);                           t_bi  = time.perf_counter() - t0

per_pair = t_all / len(all_pairs)
scale_df = pd.DataFrame([
    {"setup": f"Bi-encoder search over {len(chunks)} chunks", "seconds_per_query": round(t_bi, 4)},
    {"setup": "Bi-encoder top-10 + cross-encoder on 10",       "seconds_per_query": round(t_10, 4)},
    {"setup": f"Cross-encoder on ALL {len(chunks)} chunks",    "seconds_per_query": round(t_all, 3)},
    {"setup": "Cross-encoder on 100,000 chunks (extrapolated)", "seconds_per_query": round(per_pair * 1e5, 1)},
    {"setup": "Cross-encoder on 10,000,000 chunks (extrapolated)", "seconds_per_query": round(per_pair * 1e7, 1)},
])
scale_df

,setup,seconds_per_query
0,Bi-encoder search over 349 chunks,0.0168
1,Bi-encoder top-10 + cross-encoder on 10,0.4531
2,Cross-encoder on ALL 349 chunks,14.1940
3,"Cross-encoder on 100,000 chunks (extrapolated)",4067.1000
4,"Cross-encoder on 10,000,000 chunks (extrapolated)",406714.5000


## Step 9 — Save results for the report

In [17]:
os.makedirs("results", exist_ok=True)
rank_df.to_csv("results/rank_shifts.csv", index=False)
answers_df.to_csv("results/answers.csv", index=False)
latency_df.to_csv("results/latency.csv", index=False)
scale_df.to_csv("results/scaling.csv", index=False)
with open("results/config.json", "w") as f:
    json.dump({"bi_encoder": BI_ENCODER_NAME, "cross_encoder": CROSS_ENCODER_NAME, "llm": LLM_BACKEND,
               "chunk_words": CHUNK_WORDS, "overlap": CHUNK_OVERLAP, "num_chunks": len(chunks),
               "top_k": TOP_K, "final_k": FINAL_K, "device": DEVICE}, f, indent=2)
print(rank_df[["id", "category", "baseline_rank", "reranked_rank", "verdict"]].to_string(index=False))
print("\nSaved to results/")

# In Colab: download results/ as a zip so it can be committed to the repo
try:
    from google.colab import files
    import shutil
    shutil.make_archive("results", "zip", "results")
    files.download("results.zip")
except ImportError:
    pass

 id                category  baseline_rank  reranked_rank                                          verdict
 Q1            Keyword trap            2.0            1.0                                Both pipelines OK
 Q2            Keyword trap            3.0            3.0                                Both pipelines OK
 Q3            Keyword trap            1.0            1.0                                Both pipelines OK
 Q4 Exception / corner case            NaN            NaN Gold not in top-10 (re-ranker cannot recover it)
 Q5 Exception / corner case            NaN            NaN Gold not in top-10 (re-ranker cannot recover it)
 Q6 Exception / corner case           10.0            1.0                                 RE-RANK FIXED IT
 Q7   Negation / constraint            1.0            1.0                                Both pipelines OK
 Q8   Negation / constraint            4.0            1.0                                 RE-RANK FIXED IT
 Q9   Negation / role limit          

## Notes

* If fewer than two queries show **RE-RANK FIXED IT**, try `CHUNK_WORDS = 120` (smaller chunks separate the general rule
  from its exception more often) and re-run from Step 2, or add your own queries to `QUERIES`.
* A row marked *Gold not in top-10* is also worth reporting: re-ranking can only re-order what the first stage found.